In [1]:
import numpy as np
import matplotlib.pyplot as plt 
import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense 
from IPython.display import display, Markdown, Latex
from sklearn.datasets import make_blobs
from matplotlib.widgets import Slider

import logging 
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

## Softmax Function

In both softmax regression and neural networks with Softmax outputs, N outputs are generated and one output is selected as the predicted category. In both cases a vector **z** is generated by a linear function which is applied to a softmax function. The softmax function converts **z** into a probability distribution as described below. After applying softmax, each output will be between 0 and 1 and the outputs will add to 1, so that they can be interpreted as probabilities. The larger inputs will correspond to larger output probabilities. 

The softmax function can be written: 
$$a_j = \frac{e^{z_{j}}}{\sum\limits_{k = 1}^{N}{e^{z_k}}}\tag{1}$$
The output **a** is a vector of length N, so for softmax regression, we could also write
\begin{align}
\mathbf{a}{x} = 
\begin{bmatrix}
P(y = 1 | \mathbf{x}; \mathbf{w}, b)\\
\vdots \\ 
P(y = N | \mathbf{x}; \mathbf{w}, b)
\end{bmatrix}
=  
\frac{1}{\sum_{k = 1}^{N}{e^{z_k}}}
\begin{bmatrix}
e^{z_1} \\
\vdots \\
e^{z_{N}} \\
\end{bmatrix} \tag{2}
\end{align}

In [2]:
def my_softmax(z): 
    ez = np.exp(z) 
    sm = ez / np.sum(ez) 
    return (sm)

There are a few things to note: 
* the exponential in the numerator of the softmax magnifies small differences in the value
* the ouput vlaues sum to one
* the softmax spans all the outputs. A change in `z0` for example will change the values of `a0`-`a3`. Compared this to other activations such as ReLU or Sigmoid which have a single input and single outptu.

In [12]:
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
X_train, y_train = make_blobs(n_samples=2000, centers = centers, cluster_std=1.0, random_state=30)

## The Obvious organization

The model below is implementeed with the softmax activation in the final Dense layer. The loss function is separately specified in the `compile` directive. 

The loss function is `SparseCategoricalCrossentEntropy`. This loss is described in (3) above. In this model, the softmax takes place the last layer. The loss function takes in the softmax outptu which is a vector of probabilities. 

In [15]:
model = Sequential(
    [
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'softmax')
    ]
)

model.compile(
    loss = tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer = tf.keras.optimizers.Adam(0.001),
)

model.fit (
    X_train, y_train, 
    epochs = 10
)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 397us/step - loss: 1.2973 
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 444us/step - loss: 0.5548
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 291us/step - loss: 0.2237
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 282us/step - loss: 0.1134
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 289us/step - loss: 0.0763
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 285us/step - loss: 0.0542
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 301us/step - loss: 0.0514
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 301us/step - loss: 0.0398
Epoch 9/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 308us/step - loss: 0.0355
Epoch 10/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 309us/step - loss: 0.0371


In [17]:
p_nonpreferred = model.predict(X_train)
print(p_nonpreferred[:2])
print("largest model", np.max(p_nonpreferred), "smallest value", np.min(p_nonpreferred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 334us/step
[[9.6471747e-04 3.7642990e-03 9.8265952e-01 1.2611414e-02]
 [9.9223167e-01 7.6705967e-03 2.9708524e-05 6.7959991e-05]]
largest model 0.9999999 smallest value 8.689241e-09


### Preferred 

Recall from the lecture, more stable and accurate results can be obtained if the softmax and loss are combined during training. This is enabled by the 'preferred' organization shown here. 

In the prefered organization the final layer has a linear activation. For historical reasons, the outputs in this form are refered to as *logits*. The loss function has an additional argument `from_logits = True`. This informs the loss function that the softmax operation should be included in the loss calculation. 

In [19]:
preferred_model = Sequential(
    [
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'linear')
    ]
)

preferred_model.compile(
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer = tf.keras.optimizers.Adam(0.001),
)

preferred_model.fit (
    X_train, y_train, 
    epochs = 10
)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 365us/step - loss: 1.3038 
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 305us/step - loss: 0.6347
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 260us/step - loss: 0.3367
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 263us/step - loss: 0.1455
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 285us/step - loss: 0.0827
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 302us/step - loss: 0.0546
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 289us/step - loss: 0.0505
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 294us/step - loss: 0.0381
Epoch 9/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 302us/step - loss: 0.0374
Epoch 10/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 294us/step - loss: 0.0438


In [21]:
p_prefered = preferred_model.predict(X_train)
print(f"two example output vectors: \n {p_prefered[:2]}")
print("largest value", np.max(p_prefered), "smallest value", np.min(p_prefered))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 326us/step
two example output vectors: 
 [[-3.5570126  -1.5421472   3.7122965  -0.65276706]
 [ 7.854396    1.9926418  -1.0758071  -4.061509  ]]
largest value 13.486005 smallest value -10.537315


In [22]:
sm_prefered = tf.nn.softmax(p_prefered).numpy()
print(f"two example output vectors: \n {sm_prefered[:2]}")
print("largest value", np.max(sm_prefered), "smallest value", np.min(sm_prefered))

two example output vectors: 
 [[6.8384979e-04 5.1286807e-03 9.8170620e-01 1.2481261e-02]
 [9.9702370e-01 2.8377760e-03 1.3193737e-04 6.6633665e-06]]
largest value 0.99999857 smallest value 1.7452567e-09


In [23]:
for i in range(5): 
    print(f"{p_prefered[i]}, category: {np.argmax(p_prefered[i])}")

[-3.5570126  -1.5421472   3.7122965  -0.65276706], category: 2
[ 7.854396   1.9926418 -1.0758071 -4.061509 ], category: 0
[ 5.618058   1.7868917 -1.071857  -3.3474555], category: 0
[-2.9607005  3.679315  -1.8354067 -2.2155657], category: 1
[-0.9499608  -0.72712123  4.932328   -2.7593164 ], category: 2


### SparseCategoricalCrossentropy or CategoricalCrossEntropy

Tensorflow has two potential formates for target values and selection of the loss defines which is expected. 
* SparseCategoricalCrossentropy: expects the target to a an integer corresponding to the index. For example, if there are 10 potential target values, y would be between 0 and 9.
* CategoricalCrossEntropy: Expects the target value of an example to be one-hot encoded where the value at the target index is 1 while the other N - 1 entries are zero. An example with 10 potential target values, where the target is 2 would be [0, 0, 1, 0, 0, 0, 0,...]